Assume control outcomes at time $t\in 1,\ldots, T$ for unit $i\in1,\ldots, N$ are generated according to:
$$
\vec{y}_{it} = \mathbf{\Lambda}_i^\top \vec{f_t} + \vec{\varepsilon}_{it}.
$$ 
Let the matrix $\mathbf{\Lambda}_i = [\vec{\lambda}_{i1},\ldots,\vec{\lambda}_{iM}]$, where the $\lambda$ are column vectors, and $M$ is the number of outcomes. Note: $\vec{y}_{it}\in \mathbb{R}^{M\times 1}$, $\mathbf{\Lambda}_i \in \mathbb{R}^{k\times M}$, $\vec{f}_t\in\mathbb{R}^{k\times 1}$, and $\vec{\varepsilon}_{it}\in \mathbb{R}^{M\times 1}$, where $k$ is the number of factors. Define the matrix of outcomes at time $t$ as:
$$
\mathbf{Y}_t = (\vec{y}_{1t},\ldots, \vec{y}_{Nt})^\top = (\vec{f}_t^\top \mathbf{\Lambda}_1,\ldots, \vec{f}_t^\top \mathbf{\Lambda}_N)^\top= \begin{pmatrix}
    y_{11,t}&\ldots & y_{1M,t} \\
    \vdots &&\vdots \\
    y_{N1,t} &\ldots &y_{NM,t}
\end{pmatrix}\in \mathbb{R}^{N\times M}.
$$ 
Let $\tilde{\mathbf{Y}}_t$ denote the collection of control unit outcomes: $(\vec{y}_{2t},\ldots, \vec{y}_{Nt})^\top$.


In [ ]:
import torch
import learnQorthogonal as lqOrth
torch.manual_seed(212)

# number of time points and units
T = 7
N = 8

# number of outcomes and factors
M = 2
K = 3

In [ ]:
import torch
import learnQorthogonal as lqOrth
import numpy as np
rng = np.random.default_rng(seed=40)
torch.manual_seed(212)

# number of time points and units
T = 7
N = 8

# number of factors
K = 3


# --------------------- Factors ---------------------
def generate_factors():
    scale = 0.1
    # lists of normal rvs
    eps1 = rng.normal(0, scale, size=T)
    eps2 = rng.normal(0, scale, size=T)
    eps3 = rng.normal(0, scale, size=T)

    f1, f2, f3 = np.zeros(T), np.zeros(T), np.zeros(T)

    for t in range(T):
        # from Rong's suggestion
        f1[t] = eps1[t] + (0.8*f1[t-1] if t >= 1 else 0)
        f2[t] = eps2[t] + (-0.6*f2[t-1] + 0.8*eps2[t-1] if t >= 1 else 0)
        f3[t] = eps3[t] + (0.9*eps3[t-1] if t >= 1 else 0) + (0.4*eps3[t-2] if t >= 2 else 0)
    return f1, f2, f3

#-----------------------------------------------------

# --------------------- Loadings -------------------------------

# a function to create loadings
def generate_loadings(m):
    rho = 0.9
    Sigma = (1 - rho) * np.eye(m) + rho * np.ones((m, m))

    # list in which to collect the loadings
    loadings_all = []

    # mean vector for each factor - length of this vector is M, and we have K of them
    # mean vector is quite similar for each K 
    means = [rng.normal(0,0.5, size=m) for _ in range(K)]
    # for each unit, draw loadings for each factor
    for i in range(N):
        # the loadings for unit i by the end of the loop.
        loadings = [] # K lists of length M
        for k in range(K):
            # draw loadings from normal distribution with mean mean_k
            lambda_ik = rng.multivariate_normal(mean=means[k], cov=Sigma) 
            loadings.append(lambda_ik)
        # add the loadings for unit i to the collection of all loadings.
        loadings_all.append(loadings)
    
    return loadings_all
#--------------------------------------------------------------


# --------------------- Noiseless Outcomes ---------------------

# function to multiply factors by loadings
def generate_noiseless_outcomes(loadings_all, f1,f2,f3):
    # extract the factors over time
    factors = [np.array([f1[t], f2[t], f3[t]]) for t in range(T)]
    # to store all outcomes
    outcomes = []

    # for each unit
    for i in range(N):
        # initialize list to store outcomes
        outcomes_i = []
        # calculate the outcomes at time t
        for t in range(T):
            # the outcome at time t is the factors at t times the loadings
            f_t = factors[t]
            # K x M (3 x 2). So Lambda_i.T = (2,3). f1[t],f2[t],f3[t] is (3,)?
            Lambda_i = np.array(loadings_all[i]) 
            Y_it = np.dot(Lambda_i.T, f_t) 
            # print(f"{i},{t}, {Y_it}")
            outcomes_i.append(Y_it)
        # add the outcomes for unit i to the collection of all outcomes
        outcomes.append(outcomes_i)
    # transform to np array
    outcomes=np.array(outcomes)

    # turn the outcomes in np arrays
    for i in range(len(outcomes)):
        outcomes[i] = torch.tensor(outcomes[i]) # shape (T, M)

    # print("all outcomes:", outcomes)
    return outcomes

#-------------------------------------------------------------------------

#------------------ Shape the outcomes to be fed to the model ------------
def reshape_data(outcomes):
    # split outcomes into treated outcomes and control outcomes
    treated = torch.tensor(outcomes[0]) # (1, T, M)
    control = torch.tensor(outcomes[1:]) # (N-1, T, M)

    # break outcomes down into observations at each time point:
    treated_outcomes_list = []
    for t in range(T):
        treated_t = treated[t, :].unsqueeze(1) # (M, 1), <class 'torch.Tensor'>
        treated_outcomes_list.append(treated_t)

    # print("treated:", treated_outcomes_list)

    # break control outcomes down into observations at each time point:
    control_outcomes_list = []
    for t in range(T):
        # grab the all outcomes from all control units at time t
        control_t = control[:, t, :].T # (N-1, M), <class 'torch.Tensor'>
        control_outcomes_list.append(control_t)

    return treated_outcomes_list, control_outcomes_list

    # print("control:", control_outcomes_list)


#--------------- Calculate correlation across outcomes ---------
# TURN THIS INTO A FUNCTION!!
# Similar correlation between m1-th and m2-th outcomes across units.
# if M > 1: 
#     treated_0 = treated[:, 0]
#     treated_1 = treated[:, 1]

#     mu0 = torch.mean(treated_0)
#     mu1 = torch.mean(treated_1)

#     correlation_treated = torch.dot(treated_0 - mu0, treated_1 - mu1) / ((torch.std(treated_0) * torch.std(treated_1)))
#     correlation_treated = correlation_treated/(T-1)
#     # print(correlation_treated)

#     for i in range(len(control)):
#         first_control_unit = control[i]
#         control_0 = first_control_unit[:,0]
#         control_1 = first_control_unit[:,1]

#         mu0_c = torch.mean(control_0)
#         mu1_c = torch.mean(control_1)

#         correlation_control = torch.dot(control_0 - mu0_c, control_1 - mu1_c) / ((torch.std(control_0) * torch.std(control_1)))
#         correlation_control = correlation_control/(T-1)
#         # print(correlation_control)

#------------------- Run the Model ----------------------------
bias_dict = {}
n_iter = 10

# chain together the data generating functions
def generate_data(m):
    f1,f2,f3 = generate_factors()
    treated_outcomes_list, control_outcomes_list = reshape_data(generate_noiseless_outcomes(generate_loadings(m),f1,f2,f3))
    return treated_outcomes_list, control_outcomes_list

# calculate average bias over increasing numbers of outcomes
for m in range(2,5):
    bias_list = []
    for r in range(n_iter): 
        # generate the data
        treated_outcomes_list, control_outcomes_list = generate_data(m)
        # fit the model
        Q_final, w_final = lqOrth.learnQorthogonal(
            # only train on pre-treatment data
            targets = treated_outcomes_list[:T-1],      
            covariates = control_outcomes_list[:T-1],
            embedding_dim = 4,
            n_iterations = 2000,
            reg_Q = 0,
            reg_w = 0,
            verbose = False,
            init_Q = "eye"
        )

        # calculate the bias for this iteration
        true = treated_outcomes_list[T-1].squeeze()
        pred = (control_outcomes_list[T-1] @ Q_final @ w_final).squeeze()
        rmse_bias = torch.sqrt(torch.mean((true - pred)**2)).item()
        bias_list.append(rmse_bias)

    # calculate mean bias
    mean_bias = sum(bias_list) / len(bias_list)
    bias_dict[m] = mean_bias

print(bias_dict)

In [132]:
#------------------ Ensure stability/existence of solution --------------------#

identifiable = False
# generate a matrix L so that LL^\top has small condition number.
L = torch.randn(M*K, N-1, dtype=torch.float64)
sym = L @ L.T
if (identifiable == True):
    if (not (N-1 >= M*K)): print("Identifiability condition not met")
    while torch.linalg.cond(sym) > 100:
        L = torch.randn(M*K, N-1, dtype=torch.float64)
        sym = L @ L.T

print("Condition number of L is ", torch.linalg.cond(L))
print("Condition number of L @ L.T is ", torch.linalg.cond(sym),"\n")
#--------------------------------------------------------------------------------#

# Breaking down the L matrix into blocks representing tilde_Lambda_m for m in 1,...,M.
# print("L matrix, each break is a tilde_Lambda_m:\n")
L_blocks = L.view(M, K, N-1)
# print(L_blocks)
# print("\n")

Lambda_list = []

# Transforming the L blocks into to the actual loadings for each control unit.
for j in range(N-1):
        Lambda = torch.zeros(K, M, dtype=torch.float64)
        for i in range(M):
            Lambda[:, i:i+1] = L_blocks[i][:, j:j+1]
        Lambda_list.append([Lambda])

# The loadings for the treated unit:
Lambda_1 = torch.randn(K, M, dtype=torch.float64)

# start off with a base factor that increases linearly in T
factor_0 = torch.randn(K,1,dtype=torch.float64)
factors = [factor_0 + _*0.1*T for _ in range(T)]

# The control outcomes for time t=1,...,T_0. They are in dimension N-1 times M, ala \tilde{Y}_t
control_outcomes_list = []
for t in range(T):
    # multiply the transposed loadings by the factors at time t.
    control_outcomes_t = [torch.as_tensor(Lambda_list[i][0]).T @ factors[t] for i in range(len(Lambda_list))]
    control_outcomes_t = torch.stack(control_outcomes_t).reshape(N-1,M)
    control_outcomes_t = control_outcomes_t.T
    print(control_outcomes_t)
    control_outcomes_list.append(control_outcomes_t)

# These are the treated outcomes over t. They have dimension M times 1. 
# This makes sense: we will subtract \tilde{Y}_t^\top Q w = (M, N-1) (N-1, D)(D,1) = (M,1) from it.
treated_outcomes_list = []
for t in range(T):
    # multiply the transposed loadings by the factors at time t.
    treated_outcomes_t = Lambda_1.T @ factors[t]
    treated_outcomes_list.append(treated_outcomes_t)


Q_final, w_final = lqOrth.learnQorthogonal(
    targets = treated_outcomes_list,
    covariates = control_outcomes_list,
    embedding_dim = 9,
    n_iterations = 2000,
    reg_Q = 0,
    reg_w = 0,
    verbose = False,
    init_Q = "eye"
)

print("MAX w_final\n", w_final.max())
print("Target: \n", treated_outcomes_list[T-1])
print("SDM Prediction: \n", control_outcomes_list[T-1] @ Q_final @ w_final)
print("Bias: \n", torch.linalg.vector_norm((treated_outcomes_list[T-1].squeeze() - (control_outcomes_list[T-1] @ Q_final @ w_final))).item() / M)


Condition number of L is  tensor(1.73, dtype=torch.float64)
Condition number of L @ L.T is  tensor(3.44e+17, dtype=torch.float64) 

tensor([[ 0.19,  0.19,  0.19],
        [-0.64, -0.27, -0.30]], dtype=torch.float64)
tensor([[ 0.39,  0.31,  0.71],
        [-1.46, -0.73, -0.73]], dtype=torch.float64)
tensor([[ 0.59,  0.43,  1.23],
        [-2.29, -1.19, -1.16]], dtype=torch.float64)
tensor([[ 0.79,  0.55,  1.75],
        [-3.12, -1.65, -1.59]], dtype=torch.float64)
tensor([[ 0.99,  0.68,  2.27],
        [-3.95, -2.11, -2.02]], dtype=torch.float64)
MAX w_final
 1.0
Target: 
 tensor([[-2.73],
        [ 6.09]], dtype=torch.float64)
SDM Prediction: 
 tensor([-2.26,  4.67], dtype=torch.float64)
Bias: 
 0.7477487724818229


In [ ]:
def generate_data(m):
    identifiable = False
    M = m
    L = torch.randn(M*K, N-1, dtype=torch.float64)
    sym = L @ L.T
    if (identifiable == True):
        if (not (N-1 >= M*K)): print("Identifiability condition not met")
        while torch.linalg.cond(sym) > 100:
                    L = torch.randn(M*K, N-1, dtype=torch.float64)
                    sym = L @ L.T

    # Breaking down the L matrix into blocks representing tilde_Lambda_m for m in 1,...,M.
    # print("L matrix, each break is a tilde_Lambda_m:\n")
    L_blocks = L.view(M, K, N-1)
    # print(L_blocks)
    # print("\n")

    Lambda_list = []

    # Transforming the L blocks into to the actual loadings for each control unit.
    for j in range(N-1):
            Lambda = torch.zeros(K, M, dtype=torch.float64)
            for i in range(M):
                Lambda[:, i:i+1] = L_blocks[i][:, j:j+1]
            Lambda_list.append([Lambda])

    # The loadings for the treated unit:
    Lambda_1 = torch.randn(K, M, dtype=torch.float64)

    # start off with a base factor that increases linearly in T
    factor_0 = torch.randn(K,1,dtype=torch.float64)
    factors = [factor_0 + _*0.1*T for _ in range(T)]

    # The control outcomes for time t=1,...,T_0. They are in dimension N-1 times M, ala \tilde{Y}_t
    control_outcomes_list = []
    for t in range(T):
        # multiply the transposed loadings by the factors at time t.
        control_outcomes_t = [torch.as_tensor(Lambda_list[i][0]).T @ factors[t] for i in range(len(Lambda_list))]
        control_outcomes_t = torch.stack(control_outcomes_t).reshape(N-1,M)
        control_outcomes_t = control_outcomes_t.T
        control_outcomes_list.append(control_outcomes_t)

    # These are the treated outcomes over t. They have dimension M times 1. 
    # This makes sense: we will subtract \tilde{Y}_t^\top Q w = (M, N-1) (N-1, D)(D,1) = (M,1) from it.
    treated_outcomes_list = []
    for t in range(T):
        # multiply the transposed loadings by the factors at time t.
        treated_outcomes_t = Lambda_1.T @ factors[t]
        treated_outcomes_list.append(treated_outcomes_t)
    return control_outcomes_list, treated_outcomes_list


In [ ]:
# run bias code loop for different numbers of outcomes.
bias_dict = {}
for m in range(1, 12):
    bias_list = []
    for r in range(1):
        control_outcomes_list, treated_outcomes_list = generate_data(m)

        Q_final, w_final = lqOrth.learnQorthogonal(
            targets = treated_outcomes_list,
            covariates = control_outcomes_list,
            embedding_dim = 8,
            n_iterations = 2000,
            reg_Q = 0,
            reg_w = 0,
            verbose = False,
            init_Q = "eye"
        )
        true = treated_outcomes_list[T-1].squeeze()
        pred = control_outcomes_list[T-1] @ Q_final @ w_final
        bias = (torch.linalg.vector_norm((true - pred)).item()) / m
        bias_list.append(bias)

    mean_bias = sum(bias_list) / len(bias_list)
    bias_dict[m] = mean_bias


    


